CatBoost was created to solve one of the biggest practical problems in Machine Learning:
> How can categorical features be handled effectively without extensive preprocessing and without introducing target leakage?

### Why CatBoost Was Created
Consider a real production dataset:
| Customer_ID | City   | Device  | Income | Churn |
| ----------- | ------ | ------- | ------ | ----- |
| 1           | Mumbai | Android | 50k    | Yes   |
| 2           | Delhi  | iPhone  | 80k    | No    |

Most boosting algorithms require:
```
City
Device
```
to be encoded first.

Typically:
- One-Hot Encoding
- Label Encoding
- Target Encoding

Problems:
- Huge feature explosion
- Information loss
- Target leakage risk
- Additional preprocessing pipelines

CatBoost was designed to solve these issues.

### What is CatBoost?
CatBoost is:
> A Gradient Boosting framework that natively handles categorical features while reducing target leakage and prediction shift.

Like:
- XGBoost
- LightGBM

it is based on Gradient Boosting.

But its biggest innovation is: **Native Categorical Feature Processing**

### Why Categorical Features Are Difficult
Suppose:
```
City
```
contains:
```
Mumbai
Delhi
Pune
Hyderabad
Chennai
```
Algorithms need numbers.

#### Label Encoding Problem
```
Mumbai = 1
Delhi = 2
Pune = 3
```
Creates artificial ordering.

Model may think:
```
Pune > Delhi
```
which is meaningless.

#### One-Hot Encoding Problem
```
Mumbai → [1,0,0,0,0]
Delhi  → [0,1,0,0,0]
```
Works but:

High-cardinality features create thousands of columns.

### CatBoost's Core Innovation
CatBoost uses: **Ordered Target Statistics**

Instead of naive target encoding.

### Target Encoding Refresher
Suppose:
| City   | Churn Rate |
| ------ | ---------- |
| Mumbai | 0.80       |
| Delhi  | 0.30       |

Naive encoding:
```
Mumbai → 0.80
Delhi → 0.30
```
Problem:

Entire dataset used.

This introduces:
- Target Leakage

Future information leaks into training.

### Ordered Target Encoding
CatBoost avoids leakage.

Instead:

For each row:

Only previous rows are used.

Example:
```
Row1
No history available

Row2
Uses Row1

Row3
Uses Rows1–2

Row4
Uses Rows1–3
```
No future information used.

This is one of CatBoost's biggest innovations.

### Prediction Shift Problem
Traditional boosting suffers from:
```
Train-time distribution
≠
Inference-time distribution
```
Known as: **Prediction Shift**

CatBoost introduced: **Ordered Boosting**

to reduce this issue.

### Ordered Boosting
Instead of using all previous model outputs:

CatBoost carefully constructs boosting stages using ordered permutations.

Result:
- Reduced leakage
- Better generalization
- More stable predictions

### Tree Structure Difference
#### XGBoost
Level-wise growth.

#### LightGBM
Leaf-wise growth.

#### CatBoost
Uses:

**Symmetric Trees (Oblivious Trees)**

Example:
```
Depth 1:
Feature A

Depth 2:
Feature B

Depth 3:
Feature C
```
Same split structure across the entire level.

### Why Symmetric Trees?
Advantages:
- Faster inference
- Better regularization
- Efficient memory usage

### CatBoost Workflow
```
Raw Data
 → Ordered Target Encoding
 → Ordered Boosting
 → Symmetric Tree
```

### Important Hyperparameters
#### iterations
Number of trees.
```python
iterations=1000
```

#### learning_rate
Contribution of each tree.
```python
learning_rate=0.05
```

#### depth
Tree depth.
```python
depth=6
```

#### l2_leaf_reg
Regularization strength.

#### bagging_temperature
Controls sampling randomness.

#### loss_function
Examples:
```
Logloss
RMSE
MultiClass
```

### Classification Example
```python
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    verbose=False
)

model.fit(
    X_train,
    y_train,
    cat_features=[0, 2]
)

preds = model.predict(X_test)
```
Notice:
```python
cat_features=[0, 2]
```
No manual encoding required.

### Regression Example
```python
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6
)

model.fit(
    X_train,
    y_train,
    cat_features=[1,3]
)
```

### Feature Importance
```python
model.get_feature_importance()
```

Useful for:
- Feature selection
- Model explainability

### CatBoost vs XGBoost
| Feature                    | XGBoost  | CatBoost |
| -------------------------- | -------- | -------- |
| Native categorical support | No       | Yes      |
| Target leakage protection  | Manual   | Built-in |
| Encoding required          | Usually  | No       |
| Ease of use                | Moderate | High     |

### CatBoost vs LightGBM
| Feature              | LightGBM | CatBoost        |
| -------------------- | -------- | --------------- |
| Speed                | Faster   | Slightly slower |
| Categorical handling | Good     | Excellent       |
| Leakage prevention   | Limited  | Built-in        |
| Small datasets       | Strong   | Often Stronger  |

### When to Use CatBoost
Ideal when:
- Many categorical features
- Small-to-medium datasets
- Minimal preprocessing desired
- Fast experimentation required

### When XGBoost May Be Better
- Mostly numerical data
- Highly customized optimization
- Large mature production pipelines

### Why CatBoost Became Popular
Before CatBoost:
```
Feature Engineering
↓
Encoding
↓
Leakage Prevention
↓
Boosting
```
After CatBoost:
```
Raw categorical features
↓
CatBoost
```
Much simpler workflow.